**Part 3 — Machine Learning: predict no-shows and choose the model honestly**


In [5]:
import numpy as np
import pandas as pd
import csv
import json
import requests

Task 3.1

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
df = pd.read_csv("appointments_clean.csv")

In [8]:
df["reminder_sent"] = df["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

In [9]:
# Select input features
X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
].copy()

In [10]:
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"]
)

In [11]:
y = df["no_show"]

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [13]:
print("Training set size:", len(X_train))
print("Testing set size:", len(X_test))

Training set size: 792
Testing set size: 198


In [14]:
test_no_shows = (y_test == 1).sum()

print("No-shows in test", test_no_shows)

No-shows in test 49


In [15]:
baseline_accuracy = (y_test == 0).mean()

print("Baseline Accuracy:", baseline_accuracy)
print("Baseline Accuracy (%):", baseline_accuracy * 100)

Baseline Accuracy: 0.7525252525252525
Baseline Accuracy (%): 75.25252525252525


In [16]:
print("\nFeature matrix shapes:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nFeature columns:")
print(X.columns.tolist())


Feature matrix shapes:
X_train: (792, 11)
X_test: (198, 11)
y_train: (792,)
y_test: (198,)

Feature columns:
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']


Task 3.2

In [17]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [18]:
scaler = StandardScaler()

In [19]:
X_train_s = scaler.fit_transform(X_train)

X_test_s = scaler.transform(X_test)

In [20]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train_s, y_train)

LogisticRegression(max_iter=1000)

In [21]:
y_pred = model.predict(X_test_s)

In [22]:
y_prob = model.predict_proba(X_test_s)[:, 1]

In [23]:
# Evaluate model performance
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob)

print("LOGISTIC REGRESSION RESULTS")
print("---------------------------")
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("ROC-AUC:", roc_auc)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))


# 5. Display coefficients alongside feature names
coefficients = model.coef_[0]

coefficient_table = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": coefficients
})

print("\nFEATURE COEFFICIENTS")
print("--------------------")
print(coefficient_table.to_string(index=False))

LOGISTIC REGRESSION RESULTS
---------------------------
Accuracy: 0.7878787878787878
Precision: 0.6296296296296297
Recall: 0.3469387755102041
F1 Score: 0.4473684210526316
ROC-AUC: 0.788933022873579

Confusion Matrix:
[[139  10]
 [ 32  17]]

FEATURE COEFFICIENTS
--------------------
                Feature  Coefficient
                    age    -0.253880
            distance_km     0.253448
                fee_inr     0.452761
              lead_days     0.614471
         prior_no_shows     0.649612
          reminder_sent    -0.580669
    booking_channel_App    -0.046804
  booking_channel_Phone     0.215881
booking_channel_Walk-In    -0.186364
visit_type_Consultation     0.015658
   visit_type_Follow-up    -0.015658


In [24]:
# reminder_sent odds ratios
reminder_index = list(X.columns).index("reminder_sent")

scaled_coefficient = coefficients[reminder_index]

# Odds ratio for a one-standard-deviation increase
scaled_odds_ratio = np.exp(scaled_coefficient)

# Convert coefficient back to original feature scale
original_coefficient = (
    scaled_coefficient / scaler.scale_[reminder_index]
)

# Odds ratio for reminder_sent changing from 0 to 1
original_odds_ratio = np.exp(original_coefficient)

print("\nREMINDER SENT ANALYSIS")
print("----------------------")
print("Scaled coefficient:", scaled_coefficient)
print("Odds ratio (scaled coefficient):", scaled_odds_ratio)

print("Original-scale coefficient:", original_coefficient)
print("Odds ratio (No=0 to Yes=1):", original_odds_ratio)


REMINDER SENT ANALYSIS
----------------------
Scaled coefficient: -0.5806686033659922
Odds ratio (scaled coefficient): 0.559524141750829
Original-scale coefficient: -1.1731265130909097
Odds ratio (No=0 to Yes=1): 0.3093980903124433


Task 3.3

In [25]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [26]:

# MODEL 1: DECISION TREE WITHOUT DEPTH LIMIT


tree_unlimited = DecisionTreeClassifier(
    random_state=42
)

tree_unlimited.fit(X_train, y_train)

# Predictions
train_pred_unlimited = tree_unlimited.predict(X_train)
test_pred_unlimited = tree_unlimited.predict(X_test)

# Accuracy
train_acc_unlimited = accuracy_score(
    y_train, train_pred_unlimited
)

test_acc_unlimited = accuracy_score(
    y_test, test_pred_unlimited
)

In [27]:
# MODEL 2: DECISION TREE WITH MAX_DEPTH=3

tree_depth3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

tree_depth3.fit(X_train, y_train)

# Predictions
train_pred_depth3 = tree_depth3.predict(X_train)
test_pred_depth3 = tree_depth3.predict(X_test)

# Accuracy
train_acc_depth3 = accuracy_score(
    y_train, train_pred_depth3
)

test_acc_depth3 = accuracy_score(
    y_test, test_pred_depth3
)

In [28]:
# RESULTS

print("DECISION TREE: UNLIMITED DEPTH")
print("--------------------------------")
print("Training Accuracy:", train_acc_unlimited)
print("Test Accuracy:", test_acc_unlimited)

print("\nDECISION TREE: MAX DEPTH = 3")
print("--------------------------------")
print("Training Accuracy:", train_acc_depth3)
print("Test Accuracy:", test_acc_depth3)

DECISION TREE: UNLIMITED DEPTH
--------------------------------
Training Accuracy: 1.0
Test Accuracy: 0.6717171717171717

DECISION TREE: MAX DEPTH = 3
--------------------------------
Training Accuracy: 0.821969696969697
Test Accuracy: 0.7777777777777778


Task 3.4

In [29]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, roc_auc_score

In [30]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [31]:
rf_predictions = rf.predict(X_test)
rf_probabilities = rf.predict_proba(X_test)[:, 1]

In [32]:
# evaluation metrics
rf_accuracy = accuracy_score(y_test, rf_predictions)
rf_recall = recall_score(y_test, rf_predictions)
rf_roc_auc = roc_auc_score(y_test, rf_probabilities)

print("Random Forest Results")
print("---------------------")
print("Accuracy:", round(rf_accuracy, 4))
print("Recall:", round(rf_recall, 4))
print("ROC-AUC:", round(rf_roc_auc, 4))

Random Forest Results
---------------------
Accuracy: 0.7475
Recall: 0.3061
ROC-AUC: 0.7057


In [33]:
# Finding top three important features
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nTop 3 Important Features:")
print(feature_importance.head(3).to_string(index=False))



Top 3 Important Features:
    Feature  Importance
  lead_days    0.219905
        age    0.196788
distance_km    0.194739


Task 3.5

In [34]:
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline

Part a

In [35]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
}

In [36]:
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="roc_auc"
)


In [37]:
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validated ROC-AUC:", grid_search.best_score_)


Best parameters: {'max_depth': 3, 'n_estimators': 200}
Best cross-validated ROC-AUC: 0.7673242251739405


In [38]:

# Logistic regression pipeline
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# Other candidate models
tree_depth3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)


tuned_rf = grid_search.best_estimator_


# Calculate 5-fold cross-validated ROC-AUC
lr_cv_auc = cross_val_score(
    lr_pipe, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

tree_cv_auc = cross_val_score(
    tree_depth3, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

rf_cv_auc = cross_val_score(
    rf, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

tuned_rf_cv_auc = grid_search.best_score_


# Compare results
cv_results = {
    "Logistic Regression": lr_cv_auc,
    "Depth-3 Decision Tree": tree_cv_auc,
    "Random Forest (100 trees)": rf_cv_auc,
    "Tuned Random Forest": tuned_rf_cv_auc
}

cv_table = pd.DataFrame({
    "Model": list(cv_results.keys()),
    "Cross-validated ROC-AUC": list(cv_results.values())
})

print(cv_table.to_string(index=False))


# Select the model with the highest CV ROC-AUC
best_model_name = max(cv_results, key=cv_results.get)

model_options = {
    "Logistic Regression": lr_pipe,
    "Depth-3 Decision Tree": tree_depth3,
    "Random Forest (100 trees)": rf,
    "Tuned Random Forest": tuned_rf
}

best_model = model_options[best_model_name]

print("\nSelected model:", best_model_name)
print("Highest CV ROC-AUC:", cv_results[best_model_name])


                    Model  Cross-validated ROC-AUC
      Logistic Regression                 0.773714
    Depth-3 Decision Tree                 0.678449
Random Forest (100 trees)                 0.746602
      Tuned Random Forest                 0.767324

Selected model: Logistic Regression
Highest CV ROC-AUC: 0.7737139899948249


Part b

In [39]:
# Logistic regression pipeline
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

In [40]:
# Other candidate models
tree_depth3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

In [41]:
tuned_rf = grid_search.best_estimator_

In [42]:
# 5-fold cross-validated ROC-AUC
lr_cv_auc = cross_val_score(
    lr_pipe, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

tree_cv_auc = cross_val_score(
    tree_depth3, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

rf_cv_auc = cross_val_score(
    rf, X_train, y_train,
    cv=5, scoring="roc_auc"
).mean()

tuned_rf_cv_auc = grid_search.best_score_

In [43]:
# results comparison
cv_results = {
    "Logistic Regression": lr_cv_auc,
    "Depth-3 Decision Tree": tree_cv_auc,
    "Random Forest (100 trees)": rf_cv_auc,
    "Tuned Random Forest": tuned_rf_cv_auc
}

cv_table = pd.DataFrame({
    "Model": list(cv_results.keys()),
    "Cross-validated ROC-AUC": list(cv_results.values())
})

print(cv_table.to_string(index=False))


                    Model  Cross-validated ROC-AUC
      Logistic Regression                 0.773714
    Depth-3 Decision Tree                 0.678449
Random Forest (100 trees)                 0.746602
      Tuned Random Forest                 0.767324


In [44]:
# Select the model with the highest CV ROC-AUC
best_model_name = max(cv_results, key=cv_results.get)

model_options = {
    "Logistic Regression": lr_pipe,
    "Depth-3 Decision Tree": tree_depth3,
    "Random Forest (100 trees)": rf,
    "Tuned Random Forest": tuned_rf
}

best_model = model_options[best_model_name]

print("\nSelected model:", best_model_name)
print("Highest CV ROC-AUC:", cv_results[best_model_name])


Selected model: Logistic Regression
Highest CV ROC-AUC: 0.7737139899948249


part c

In [51]:
oof_prob = cross_val_predict(
    best_model,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

In [52]:
# Compare thresholds
thresholds = [0.50, 0.35]

threshold_results = []

for threshold in thresholds:

    # Flag as no-show risk if probability >= threshold
    oof_pred = (oof_prob >= threshold).astype(int)

    precision = precision_score(
        y_train, oof_pred, zero_division=0
    )

    recall = recall_score(
        y_train, oof_pred, zero_division=0
    )

    f1 = f1_score(
        y_train, oof_pred, zero_division=0
    )

    number_flagged = oof_pred.sum()

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Number Flagged": number_flagged
    })


threshold_table = pd.DataFrame(threshold_results)

print("THRESHOLD COMPARISON")
print(threshold_table.to_string(index=False))

THRESHOLD COMPARISON
 Threshold  Precision   Recall       F1  Number Flagged
      0.50   0.575342 0.248521 0.347107              73
      0.35   0.512987 0.467456 0.489164             154


In [53]:
# Choosing threshold with the higher F1 score
best_threshold = threshold_table.loc[
    threshold_table["F1"].idxmax(),
    "Threshold"
]

print("\nSelected threshold:", best_threshold)



Selected threshold: 0.35


part d

In [54]:
best_model.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=1000))])

In [55]:
test_prob = best_model.predict_proba(X_test)[:, 1]

In [56]:
test_pred = (test_prob >= best_threshold).astype(int)

In [57]:
test_precision = precision_score(
    y_test, test_pred, zero_division=0
)

In [58]:
test_recall = recall_score(
    y_test, test_pred, zero_division=0
)

In [59]:
test_roc_auc = roc_auc_score(y_test, test_prob)

number_flagged = test_pred.sum()

In [60]:
print("FINAL TEST SET RESULTS")
print("----------------------")
print("Selected model:", best_model_name)
print("Selected threshold:", best_threshold)
print("Precision:", test_precision)
print("Recall:", test_recall)
print("Number flagged:", number_flagged)
print("ROC-AUC:", test_roc_auc)


FINAL TEST SET RESULTS
----------------------
Selected model: Logistic Regression
Selected threshold: 0.35
Precision: 0.5714285714285714
Recall: 0.5714285714285714
Number flagged: 49
ROC-AUC: 0.788933022873579


part e

In [61]:
test_appointment_ids = df.loc[
    X_test.index, "appointment_id"
].to_numpy()

In [62]:
risk_scores = pd.DataFrame({
    "appointment_id": test_appointment_ids,
    "no_show": y_test.to_numpy(),
    "risk_prob": test_prob,
    "flag": test_pred
})

In [63]:
risk_scores.to_csv(
    "test_risk_scores.csv",
    index=False
)

In [64]:
print("Saved test_risk_scores.csv")
print(risk_scores.head())
print("\nTotal test rows:", len(risk_scores))

Saved test_risk_scores.csv
  appointment_id  no_show  risk_prob  flag
0          A0217        0   0.021565     0
1          A0336        0   0.150917     0
2          A0508        0   0.518692     1
3          A0314        0   0.191300     0
4          A0089        0   0.167718     0

Total test rows: 198
